Logit Regression Models

# Data import, loading libraries, set up logging et cetera

## Import data

In [13]:
from google.cloud import bigquery as bq
import pandas as pd

bq_client = bq.Client()

def load_bq_table_to_dataframe(table_path: str):
    """
    Loads data from a specified BigQuery table into a pandas DataFrame.

    Args:
        table_path (str): The full path to the BigQuery table (e.g., 'project.dataset.table').

    Returns:
        pandas.DataFrame: A DataFrame containing the data from the BigQuery table.
    """
    query_string = f"SELECT * FROM `{table_path}`"
    print(f"Executing BigQuery query for table: {table_path}")
    bq_query_job = bq_client.query(query_string, job_config=bq.QueryJobConfig())
    return bq_query_job.to_dataframe()

# Define the table paths (as strings)
base_path = 'ingka-ff-somdata-prod.OMDA_Analytics.'
path = f'{base_path}.no_stock_deviation_predictions_arf_delivery'

# Load data frames
df = load_bq_table_to_dataframe(path)

print("DataFrames loaded successfully!")

Executing BigQuery query for table: ingka-ff-somdata-prod.OMDA_Analytics..no_stock_deviation_predictions_arf_delivery
DataFrames loaded successfully!


## Importing libraries

In [14]:
# ============================================================================
# IMPORTS
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')


## Set output path

In [15]:
import os
import logging

# Define output path
OUTPUT_PATH = 'C:/Users/NILAV/OneDrive - IKEA/Documents/Project folder/NO STOCK business case/Casual Inference/'

# Ensure output directory exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ----------------------------------------------------------------------------
# LOGGING SETUP
# ----------------------------------------------------------------------------

log_file = os.path.join(OUTPUT_PATH, 'preprocessing_log.txt')

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='a'   # append so du kan köra skriptet flera gånger
)

def log(msg: str):
    """Logga både till konsol och till fil."""
    print(msg)
    logging.info(msg)


# Keep relevant features for analysis

In [16]:
# Columns to keep
cols_to_keep = [
    "country_code",
    "order_no",
    "service_prime_line_no",
    "first_event_time",
    "successful_AR",
    "stock_z_score_imputed",
    "initial_ship_node",
    "initial_no_order_lines",
    "initial_ordered_qty",
    "initial_total_euros",
    "initial_click_collect_service_type",
    "initial_puop",
    "initial_pup",
    "initial_home_delivery_service_type",
    "initial_is_null_service_type",
    "initial_remote_bedroom",
    "initial_chat",
    "initial_written",
    "initial_remote_email_sales",
    "initial_remote_kitchen",
    "initial_in_home_planning",
    "initial_store",
    "initial_phone",
    "initial_remote_other_range",
    "initial_digital_meeting",
    "initial_proactive_chat",
    "initial_remote_int_design",
    "initial_social_media",
    "initial_outbound_phone",
    "initial_internet",
    "initial_letter",
    "initial_telansmach",
    "initial_outbound_lead_sales",
    "initial_remote_livingroom",
    "initial_fax",
    "initial_is_null_sales_channel",
    "initial_parcel",
    "initial_truck",
    "initial_is_null_transport_method_type",
    "initial_open_invoice",
    "initial_cash_on_fulfillment",
    "initial_invoice",
    "initial_payment_on_delivery",
    "initial_cash",
    "initial_is_null_payment_type",
    "initial_bathroom_and_water_qty",
    "initial_bed_and_bath_textiles_qty",
    "initial_bedroom_furniture_qty",
    "initial_beds_and_mattresses_qty",
    "initial_children_s_ikea_qty",
    "initial_cooking_qty",
    "initial_decoration_qty",
    "initial_dining_qty",
    "initial_eating_qty",
    "initial_home_appliances_qty",
    "initial_home_organisation_qty",
    "initial_home_textiles_qty",
    "initial_kitchen_qty",
    "initial_lighting_and_home_electronics_qty",
    "initial_living_room_seating_qty",
    "initial_other_business_opportunities_qty",
    "initial_outdoor_qty",
    "initial_rugs_qty",
    "initial_store_and_organise_furniture_qty",
    "initial_workspaces_qty",
    "initial_sl1_items",
    "initial_sl2_items",
    "initial_sl3_items",
    "initial_sl4_items",
    "initial_sl_item_is_null",
    "time_to_auto_recovery_hours",
    "hours_until_promised_delivery",
    "hours_total_promised_lead_time",
]

# Optionally: check which requested columns are missing in df
missing_in_df = [c for c in cols_to_keep if c not in df.columns]
if missing_in_df:
    print("Warning: these columns were not found in df and will be skipped:")
    print(missing_in_df)

# Keep only the intersection of requested columns and existing columns
cols_existing = [c for c in cols_to_keep if c in df.columns]

# Subset df to only these columns (and keep the name df)
df = df[cols_existing].copy()

print("Final df shape:", df.shape)


Final df shape: (32350, 73)


# STEP 0: EDA

In [17]:
import os
import pandas as pd
import numpy as np

# Assume:
# - df already exists (e.g. loaded from BigQuery)
# - OUTPUT_PATH is a string with the output directory, e.g. "output/step0"
#   and you want all Excel files written there.

# Ensure output directory exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# =========================
# STEP 0A: Basic overview
# =========================

print("Shape (rows, cols):", df.shape)

print("\nDtypes:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

print("\nInfo:")
df.info()

# =========================
# STEP 0B: Missing values
# =========================

missing = (
    df.isna()
      .sum()
      .to_frame("n_missing")
      .assign(
          pct_missing=lambda x: x["n_missing"] / len(df) * 100
      )
      .sort_values("pct_missing", ascending=False)
)

print("\nMissing values per column (sorted):")
display(missing)

# Write to Excel
missing.to_excel(
    os.path.join(OUTPUT_PATH, "step0_missing_values.xlsx"),
    sheet_name="missing_values",
    index=True
)

# =========================
# STEP 0C: Identify numeric & binary columns
# =========================

numeric_cols = df.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

binary_cols = []
for col in numeric_cols:
    unique_vals = sorted(df[col].dropna().unique())
    if len(unique_vals) <= 2 and set(unique_vals).issubset({0, 1}):
        binary_cols.append(col)

print("\nNumeric columns:")
print(numeric_cols)

print("\nBinary 0/1 columns:")
print(binary_cols)

# =========================
# STEP 0D: Prevalence for binary variables
# =========================

if binary_cols:
    bin_stats = (
        df[binary_cols]
        .mean()
        .to_frame("prevalence_1")
        .assign(
            prevalence_1_pct=lambda x: x["prevalence_1"] * 100,
            n_ones=lambda x: (df[binary_cols] == 1).sum()
        )
        .sort_values("prevalence_1")
    )
    print("\nPrevalence for binary variables (share of 1's):")
    display(bin_stats)

    # Write to Excel
    bin_stats.to_excel(
        os.path.join(OUTPUT_PATH, "step0_binary_prevalence.xlsx"),
        sheet_name="binary_prevalence",
        index=True
    )
else:
    print("\nNo binary 0/1 columns detected by the simple rule.")

# =========================
# STEP 0E: Distribution for key numeric variables
# =========================

key_numeric = [
    "time_to_auto_recovery_hours",
    "hours_until_promised_delivery",
    "hours_total_promised_lead_time",
    "initial_no_order_lines",
    "initial_ordered_qty",
    "initial_total_euros",
    "stock_z_score_imputed",
]

key_numeric = [c for c in key_numeric if c in df.columns]

if key_numeric:
    desc_stats = (
        df[key_numeric]
        .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
        .T
    )
    print("\nDescriptive stats for selected numeric variables:")
    display(desc_stats)

    # Write to Excel
    desc_stats.to_excel(
        os.path.join(OUTPUT_PATH, "step0_numeric_descriptives.xlsx"),
        sheet_name="descriptives",
        index=True
    )
else:
    print("\nNone of the expected key variables were found in df.")

# =========================
# STEP 0F: Correlation between numeric variables
# =========================

corr_numeric = list(set(numeric_cols).intersection(set(key_numeric))) or numeric_cols

if len(corr_numeric) > 1:
    corr_matrix = df[corr_numeric].corr()
    print("\nCorrelation (Pearson) between numeric variables:")
    display(corr_matrix)

    # Write to Excel
    corr_matrix.to_excel(
        os.path.join(OUTPUT_PATH, "step0_numeric_correlation.xlsx"),
        sheet_name="correlation",
        index=True
    )
else:
    print("\nToo few numeric variables for a correlation matrix.")

# =========================
# STEP 0G: Ship node overview
# =========================

if "initial_ship_node" in df.columns:
    ship_counts = (
        df["initial_ship_node"]
        .value_counts()
        .to_frame("n_deliveries")
        .assign(pct=lambda x: x["n_deliveries"] / len(df) * 100)
    )
    print("\nDistribution of initial_ship_node (top 20):")
    display(ship_counts.head(20))

    print("\nTotal number of ship nodes:", ship_counts.shape[0])
    print("Ship nodes with < 100 rows:", (ship_counts["n_deliveries"] < 100).sum())

    # Write to Excel
    ship_counts.to_excel(
        os.path.join(OUTPUT_PATH, "step0_ship_node_overview.xlsx"),
        sheet_name="ship_nodes",
        index=True
    )
else:
    print("\nColumn initial_ship_node does not exist in df.")


Shape (rows, cols): (32350, 73)

Dtypes:
country_code                                   object
order_no                                       object
service_prime_line_no                          object
first_event_time                  datetime64[us, UTC]
successful_AR                                   Int64
                                         ...         
initial_sl4_items                               Int64
initial_sl_item_is_null                         Int64
time_to_auto_recovery_hours                   float64
hours_until_promised_delivery                 float64
hours_total_promised_lead_time                float64
Length: 73, dtype: object

First 5 rows:


,country_code,order_no,service_prime_line_no,first_event_time,successful_AR,stock_z_score_imputed,initial_ship_node,initial_no_order_lines,initial_ordered_qty,initial_total_euros,...,initial_store_and_organise_furniture_qty,initial_workspaces_qty,initial_sl1_items,initial_sl2_items,initial_sl3_items,initial_sl4_items,initial_sl_item_is_null,time_to_auto_recovery_hours,hours_until_promised_delivery,hours_total_promised_lead_time
0,US,483930124,6,2025-10-25 14:58:03+00:00,0,-0.223770,CDC.945,2,8.0,59.462972,...,0.0,0.0,0,0,0,2,0,0.006667,46.032500,46.039167
1,IT,1557296309,131,2025-10-31 13:55:38+00:00,0,0.148994,CDC.074,124,209.0,7817.000000,...,0.0,0.0,16,46,46,12,1,2.539444,737.072778,739.612222
2,SE,1555709876,33,2025-11-01 02:07:50+00:00,1,0.742486,CDC.405,31,100.0,2304.483060,...,0.0,0.0,2,0,11,18,0,177.233333,151.869444,329.102778
3,US,484015649,4,2025-10-27 01:53:06+00:00,0,0.132509,CDC.051,1,3.0,136.664430,...,3.0,0.0,0,0,0,0,1,0.005556,83.115000,83.120556
4,US,483157728,10,2025-10-07 20:46:38+00:00,1,-1.288874,CDC.023,5,5.0,525.237662,...,1.0,2.0,0,1,1,3,0,0.006111,43.222778,43.228889



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32350 entries, 0 to 32349
Data columns (total 73 columns):
 #   Column                                     Non-Null Count  Dtype              
---  ------                                     --------------  -----              
 0   country_code                               32350 non-null  object             
 1   order_no                                   32350 non-null  object             
 2   service_prime_line_no                      32350 non-null  object             
 3   first_event_time                           32350 non-null  datetime64[us, UTC]
 4   successful_AR                              32350 non-null  Int64              
 5   stock_z_score_imputed                      32350 non-null  float64            
 6   initial_ship_node                          32322 non-null  object             
 7   initial_no_order_lines                     32350 non-null  Int64              
 8   initial_ordered_qty                    

,n_missing,pct_missing
initial_ship_node,28,0.086553
country_code,0,0.000000
order_no,0,0.000000
first_event_time,0,0.000000
service_prime_line_no,0,0.000000
...,...,...
initial_sl4_items,0,0.000000
initial_sl_item_is_null,0,0.000000
time_to_auto_recovery_hours,0,0.000000
hours_until_promised_delivery,0,0.000000



Numeric columns:
['successful_AR', 'stock_z_score_imputed', 'initial_no_order_lines', 'initial_ordered_qty', 'initial_total_euros', 'initial_click_collect_service_type', 'initial_puop', 'initial_pup', 'initial_home_delivery_service_type', 'initial_is_null_service_type', 'initial_remote_bedroom', 'initial_chat', 'initial_written', 'initial_remote_email_sales', 'initial_remote_kitchen', 'initial_in_home_planning', 'initial_store', 'initial_phone', 'initial_remote_other_range', 'initial_digital_meeting', 'initial_proactive_chat', 'initial_remote_int_design', 'initial_social_media', 'initial_outbound_phone', 'initial_internet', 'initial_letter', 'initial_telansmach', 'initial_outbound_lead_sales', 'initial_remote_livingroom', 'initial_fax', 'initial_is_null_sales_channel', 'initial_parcel', 'initial_truck', 'initial_is_null_transport_method_type', 'initial_open_invoice', 'initial_cash_on_fulfillment', 'initial_invoice', 'initial_payment_on_delivery', 'initial_cash', 'initial_is_null_payme

...


Correlation (Pearson) between numeric variables:


,initial_total_euros,hours_until_promised_delivery,stock_z_score_imputed,time_to_auto_recovery_hours,initial_ordered_qty,initial_no_order_lines,hours_total_promised_lead_time
initial_total_euros,1.000000,0.298684,0.001214,0.385164,0.788973,0.828004,0.449659
hours_until_promised_delivery,0.298684,1.000000,0.009515,0.182545,0.269999,0.314061,0.681353
stock_z_score_imputed,0.001214,0.009515,1.000000,0.004713,0.006344,0.009096,0.008699
time_to_auto_recovery_hours,0.385164,0.182545,0.004713,1.000000,0.310450,0.375154,0.844034
initial_ordered_qty,0.788973,0.269999,0.006344,0.310450,1.000000,0.820562,0.378391
initial_no_order_lines,0.828004,0.314061,0.009096,0.375154,0.820562,1.000000,0.450595
hours_total_promised_lead_time,0.449659,0.681353,0.008699,0.844034,0.378391,0.450595,1.000000



Distribution of initial_ship_node (top 20):


,n_deliveries,pct
initial_ship_node,,
CDC.018,8852,27.363215
CDC.945,7463,23.069552
CDC.003,2136,6.602782
CDC.405,1058,3.270479
CDC.424,994,3.072643
CDC.074,928,2.868624
CDC.054,754,2.330757
CDC.051,734,2.268934
CDC.048,618,1.910355



Total number of ship nodes: 44
Ship nodes with < 100 rows: 10


In [18]:
import numpy as np
import pandas as pd

# Assumptions:
# - df is your working DataFrame
# - You want a general cleaning step that:
#   1) Drops any row that has missing values in any column
#   2) Drops binary columns that are always 0 (no variance)
#   3) Either drops or groups binary columns with extremely low prevalence

# -----------------------------
# Configuration
# -----------------------------
# Minimum number of 1's required to keep a binary column as its own feature
MIN_ONES = 20          # adjust if needed
GROUP_RARE_BINARIES = True  # if True: create one "any_rare_binary" flag; if False: just drop them

# Keep raw copy for reference
df_raw = df.copy()

# -----------------------------
# 1) Drop rows with any missing values
# -----------------------------
df = df.dropna(axis=0, how="any")

print("Rows before dropna:", df_raw.shape[0])
print("Rows after dropna: ", df.shape[0])
print("Dropped rows:      ", df_raw.shape[0] - df.shape[0])

# -----------------------------
# 2) Identify numeric and binary columns
# -----------------------------
numeric_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

binary_cols = []
for col in numeric_cols:
    unique_vals = sorted(df[col].dropna().unique())
    # Treat as binary if only 0/1 (or just {0} or {1}) exist
    if len(unique_vals) <= 2 and set(unique_vals).issubset({0, 1}):
        binary_cols.append(col)

print("\nDetected numeric columns:", len(numeric_cols))
print("Detected binary columns: ", len(binary_cols))

# -----------------------------
# 3) Compute prevalence for binary columns
# -----------------------------
if binary_cols:
    bin_stats = (
        df[binary_cols]
        .mean()
        .to_frame("prevalence_1")
        .assign(
            prevalence_1_pct=lambda x: x["prevalence_1"] * 100,
            n_ones=lambda x: (df[binary_cols] == 1).sum()
        )
        .sort_values("prevalence_1")
    )
else:
    bin_stats = pd.DataFrame(columns=["prevalence_1", "prevalence_1_pct", "n_ones"])

print("\nBinary prevalence (head):")
display(bin_stats.head(20))

# -----------------------------
# 4) Drop binary columns with only 0 (no variance)
# -----------------------------
zero_variance_binary = bin_stats[bin_stats["n_ones"] == 0].index.tolist()

print("\nBinary columns with zero variance (all zeros):")
print(zero_variance_binary)

df = df.drop(columns=zero_variance_binary, errors="ignore")

# Update binary_cols after dropping zero-variance columns
binary_cols = [c for c in binary_cols if c not in zero_variance_binary]
bin_stats = bin_stats.loc[binary_cols]

# -----------------------------
# 5) Handle rare binary columns (extremely low prevalence)
# -----------------------------
rare_binary = bin_stats[
    (bin_stats["n_ones"] < MIN_ONES) &  # too few ones
    (bin_stats["n_ones"] > 0)           # exclude all-zero (already removed)
].index.tolist()

print("\nBinary columns with very low prevalence (n_ones < MIN_ONES):")
print(rare_binary)

if GROUP_RARE_BINARIES and rare_binary:
    # Create a single flag indicating if any rare binary is 1
    df["any_rare_binary_flag"] = df[rare_binary].any(axis=1).astype(int)
    print("\nCreated 'any_rare_binary_flag' from rare binaries.")
    
    # Drop the individual rare columns
    df = df.drop(columns=rare_binary, errors="ignore")
elif rare_binary:
    # Just drop all rare binary columns
    df = df.drop(columns=rare_binary, errors="ignore")
    print("\nDropped rare binary columns without grouping.")

# Final reporting
print("\nFinal shape after cleaning:", df.shape)


Rows before dropna: 32350
Rows after dropna:  32322
Dropped rows:       28

Detected numeric columns: 68
Detected binary columns:  39

Binary prevalence (head):


,prevalence_1,prevalence_1_pct,n_ones
initial_is_null_service_type,0.0,0.0,0
initial_written,0.0,0.0,0
initial_in_home_planning,0.0,0.0,0
initial_is_null_transport_method_type,0.0,0.0,0
initial_is_null_sales_channel,0.0,0.0,0
initial_other_business_opportunities_qty,0.0,0.0,0
initial_children_s_ikea_qty,0.0,0.0,0
initial_is_null_payment_type,0.0,0.0,0
initial_social_media,0.000031,0.003094,1
initial_remote_livingroom,0.000062,0.006188,2



Binary columns with zero variance (all zeros):
['initial_is_null_service_type', 'initial_written', 'initial_in_home_planning', 'initial_is_null_transport_method_type', 'initial_is_null_sales_channel', 'initial_other_business_opportunities_qty', 'initial_children_s_ikea_qty', 'initial_is_null_payment_type']

Binary columns with very low prevalence (n_ones < MIN_ONES):
['initial_digital_meeting', 'initial_proactive_chat', 'initial_social_media', 'initial_letter', 'initial_telansmach', 'initial_remote_livingroom', 'initial_payment_on_delivery']

Created 'any_rare_binary_flag' from rare binaries.

Final shape after cleaning: (32322, 59)


# STEP 1: Feature Engineering 
* Group ship_nodes (check unique count first!)
* Aggregate HFB to 4-5 features
* Group sales channels to top 5 + other 
* Remove payment type vars 
* Create pct_sl1, has_large_furniture, etc. 


In [20]:
import numpy as np
import pandas as pd

# ================================
# STEP 1: Feature Engineering
# ================================
# This assumes df already contains the subset of columns you listed.

# ------------------------------------------------
# 1) Group ship nodes into tiers (LARGE/MEDIUM/SMALL)
# ------------------------------------------------
if "initial_ship_node" in df.columns:
    ship_counts = df["initial_ship_node"].value_counts()
    print("Ship node counts (top 10):")
    display(ship_counts.head(10))

    # Use quartiles to define tiers (adjust logic if needed)
    q25, q75 = ship_counts.quantile([0.25, 0.75])

    def map_ship_node_tier(node):
        n = ship_counts.get(node, 0)
        if n >= q75:
            return "LARGE"
        elif n >= q25:
            return "MEDIUM"
        else:
            return "SMALL"

    df["ship_node_tier"] = df["initial_ship_node"].map(map_ship_node_tier)

    print("\nShip node tier distribution:")
    display(df["ship_node_tier"].value_counts())
else:
    print("Column 'initial_ship_node' not found; skipping ship node grouping.")

# ------------------------------------------------
# 2) Aggregate HFB quantities to 4–5 features
# ------------------------------------------------
# Identify HFB quantity columns by pattern
hfb_qty_cols = [
    c for c in df.columns
    if c.startswith("initial_") and c.endswith("_qty")
]

print("\nHFB quantity columns detected:")
print(hfb_qty_cols)

if hfb_qty_cols:
    # Total HFB quantity
    df["hfb_total_qty"] = df[hfb_qty_cols].sum(axis=1)

    # Number of distinct HFB categories with >0 qty
    df["hfb_n_categories"] = (df[hfb_qty_cols] > 0).sum(axis=1)

    # Define some meaningful HFB groups
    large_furniture_cols = [
        "initial_bedroom_furniture_qty",
        "initial_living_room_seating_qty",
        "initial_store_and_organise_furniture_qty",
        "initial_workspaces_qty",
        "initial_beds_and_mattresses_qty",
    ]
    complex_items_cols = [
        "initial_kitchen_qty",
        "initial_bathroom_and_water_qty",
        "initial_home_appliances_qty",
    ]
    textiles_decor_cols = [
        "initial_bed_and_bath_textiles_qty",
        "initial_home_textiles_qty",
        "initial_rugs_qty",
        "initial_decoration_qty",
    ]

    # Keep only those that actually exist in df
    large_furniture_cols = [c for c in large_furniture_cols if c in df.columns]
    complex_items_cols = [c for c in complex_items_cols if c in df.columns]
    textiles_decor_cols = [c for c in textiles_decor_cols if c in df.columns]

    # Aggregated quantities
    if large_furniture_cols:
        df["hfb_large_furniture_qty"] = df[large_furniture_cols].sum(axis=1)
    else:
        df["hfb_large_furniture_qty"] = 0

    if complex_items_cols:
        df["hfb_complex_items_qty"] = df[complex_items_cols].sum(axis=1)
    else:
        df["hfb_complex_items_qty"] = 0

    if textiles_decor_cols:
        df["hfb_textiles_decor_qty"] = df[textiles_decor_cols].sum(axis=1)
    else:
        df["hfb_textiles_decor_qty"] = 0

    # Binary flag: any large furniture
    df["has_large_furniture"] = (df["hfb_large_furniture_qty"] > 0).astype(int)

else:
    print("No HFB *_qty columns found; skipping HFB aggregation.")

# ------------------------------------------------
# 3) Group sales channels to top N + 'other'
# ------------------------------------------------
# Explicit list of sales channel dummy columns (from your variable list)
sales_channel_cols = [
    "initial_remote_bedroom",
    "initial_chat",
    "initial_written",
    "initial_remote_email_sales",
    "initial_remote_kitchen",
    "initial_in_home_planning",
    "initial_store",
    "initial_phone",
    "initial_remote_other_range",
    "initial_digital_meeting",
    "initial_proactive_chat",
    "initial_remote_int_design",
    "initial_social_media",
    "initial_outbound_phone",
    "initial_internet",
    "initial_letter",
    "initial_telansmach",
    "initial_outbound_lead_sales",
    "initial_remote_livingroom",
    "initial_fax",
    "initial_is_null_sales_channel",
]

# Keep only those that exist in df
sales_channel_cols = [c for c in sales_channel_cols if c in df.columns]

if sales_channel_cols:
    # Compute counts per channel
    sales_counts = df[sales_channel_cols].sum().sort_values(ascending=False)
    print("\nSales channel counts:")
    display(sales_counts)

    # Choose top N channels to keep as separate dummies
    TOP_N = 5
    top_channels = sales_counts.head(TOP_N).index.tolist()

    # Exclude "null" channel from grouping into "other" (treat as separate)
    null_channel = "initial_is_null_sales_channel"
    if null_channel in top_channels:
        # Keep it in top_channels, but do not include it in "other" logic
        pass

    # Non-top channels (excluding possible null channel)
    other_channels = [
        c for c in sales_channel_cols
        if c not in top_channels and c != null_channel
    ]

    print("\nTop sales channels kept as individual dummies:")
    print(top_channels)
    print("\nSales channels grouped into 'other':")
    print(other_channels)

    # Create "other" flag if there are any grouped channels
    if other_channels:
        df["initial_sales_channel_other"] = df[other_channels].any(axis=1).astype(int)
    else:
        df["initial_sales_channel_other"] = 0

    # Keep top channels + null channel + 'other', drop the rest
    cols_to_keep_sc = set(top_channels)
    if null_channel in df.columns:
        cols_to_keep_sc.add(null_channel)
    cols_to_keep_sc.add("initial_sales_channel_other")

    # Determine which sales channel columns to drop
    cols_to_drop_sc = [
        c for c in sales_channel_cols
        if c not in cols_to_keep_sc
    ]

    df = df.drop(columns=cols_to_drop_sc, errors="ignore")

    print("\nAfter grouping, sales channel columns in df:")
    print([c for c in df.columns if c.startswith("initial_") and "sales_channel" in c or c in top_channels])
else:
    print("No sales channel columns found; skipping sales channel grouping.")

# ------------------------------------------------
# 4) Remove payment type variables
# ------------------------------------------------
payment_type_cols = [
    "initial_open_invoice",
    "initial_cash_on_fulfillment",
    "initial_invoice",
    "initial_payment_on_delivery",
    "initial_cash",
    "initial_is_null_payment_type",
]

payment_type_cols = [c for c in payment_type_cols if c in df.columns]

if payment_type_cols:
    print("\nDropping payment type columns:")
    print(payment_type_cols)
    df = df.drop(columns=payment_type_cols, errors="ignore")
else:
    print("No payment type columns found; skipping payment type removal.")

# ------------------------------------------------
# 5) Create pct_sl1, has_large_furniture, and related service level features
# ------------------------------------------------
sl_cols = [
    "initial_sl1_items",
    "initial_sl2_items",
    "initial_sl3_items",
    "initial_sl4_items",
]

sl_cols = [c for c in sl_cols if c in df.columns]

if sl_cols:
    # Total items with defined service level (SL1–SL4)
    df["initial_sl_total_items"] = df[sl_cols].sum(axis=1)

    # Share of SL1 items among all SL1–SL4 items
    # Guard against division by zero by replacing 0 with NaN then filling with 0
    df["pct_sl1_items"] = (
        df["initial_sl1_items"] / df["initial_sl_total_items"].replace(0, np.nan)
    ).fillna(0.0)

    # Binary flag: any SL1 items in the delivery
    df["has_sl1_items"] = (df["initial_sl1_items"] > 0).astype(int)

    # Optionally: share of high-priority items (SL1+SL2)
    if "initial_sl2_items" in df.columns:
        df["pct_high_priority_items"] = (
            (df["initial_sl1_items"] + df["initial_sl2_items"])
            / df["initial_sl_total_items"].replace(0, np.nan)
        ).fillna(0.0)
else:
    print("No service level columns (SL1–SL4) found; skipping SL features.")

print("\nFeature engineering STEP 1 completed. Current df shape:", df.shape)


Ship node counts (top 10):


initial_ship_node
CDC.018    8852
CDC.945    7463
CDC.003    2136
CDC.405    1058
CDC.424     994
CDC.074     928
CDC.054     754
CDC.051     734
CDC.048     618
CDC.960     612
Name: count, dtype: int64


Ship node tier distribution:


ship_node_tier
LARGE     24750
MEDIUM     6856
SMALL       716
Name: count, dtype: int64


HFB quantity columns detected:
['initial_ordered_qty', 'initial_bathroom_and_water_qty', 'initial_bed_and_bath_textiles_qty', 'initial_bedroom_furniture_qty', 'initial_beds_and_mattresses_qty', 'initial_cooking_qty', 'initial_decoration_qty', 'initial_dining_qty', 'initial_eating_qty', 'initial_home_appliances_qty', 'initial_home_organisation_qty', 'initial_home_textiles_qty', 'initial_kitchen_qty', 'initial_lighting_and_home_electronics_qty', 'initial_living_room_seating_qty', 'initial_outdoor_qty', 'initial_rugs_qty', 'initial_store_and_organise_furniture_qty', 'initial_workspaces_qty']

Sales channel counts:


initial_internet               25336
initial_store                   4161
initial_phone                   1693
initial_remote_kitchen           363
initial_remote_email_sales       205
initial_chat                     176
initial_outbound_phone           112
initial_remote_bedroom            63
initial_outbound_lead_sales       60
initial_fax                       39
initial_remote_int_design         27
initial_remote_other_range        24
dtype: Int64


Top sales channels kept as individual dummies:
['initial_internet', 'initial_store', 'initial_phone', 'initial_remote_kitchen', 'initial_remote_email_sales']

Sales channels grouped into 'other':
['initial_remote_bedroom', 'initial_chat', 'initial_remote_other_range', 'initial_remote_int_design', 'initial_outbound_phone', 'initial_outbound_lead_sales', 'initial_fax']

After grouping, sales channel columns in df:
['initial_remote_email_sales', 'initial_remote_kitchen', 'initial_store', 'initial_phone', 'initial_internet', 'initial_sales_channel_other']

Dropping payment type columns:
['initial_open_invoice', 'initial_cash_on_fulfillment', 'initial_invoice', 'initial_cash']

Feature engineering STEP 1 completed. Current df shape: (32322, 60)


# STEP 2: Data Prep 
* Convert to int/float 
* Create dummies (DROP ONE for each category!) 
* Verify no perfect collinearity
* Create df_modeling

In [21]:
import numpy as np
import pandas as pd

# =======================================
# STEP 2: Data Preparation
# - Convert to numeric where appropriate
# - Create dummies for categorical variables (drop one per category)
# - Verify no perfect collinearity
# =======================================

# Keep a copy of the original df for reference
df_raw_step2 = df.copy()

# ------------------------------------------------
# 2.1 Convert to int/float where appropriate
# ------------------------------------------------
# Identify object columns that might actually contain numeric values (e.g. "0", "1", "123.45")
object_cols = df.select_dtypes(include=["object"]).columns.tolist()
print("Object columns before numeric conversion:", object_cols)

# If you know some object columns are truly categorical (e.g. 'country_code'),
# you can exclude them from numeric conversion:
categorical_like = ["country_code", "initial_ship_node"]  # adjust as needed
cols_try_numeric = [c for c in object_cols if c not in categorical_like]

for col in cols_try_numeric:
    # Try converting to numeric; non-numeric values become NaN
    df[col] = pd.to_numeric(df[col], errors="ignore")

# Safe conversion of boolean columns to int (if any)
bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
for col in bool_cols:
    df[col] = df[col].astype(int)

print("\nDtypes after numeric conversion attempts:")
print(df.dtypes)


# ------------------------------------------------
# 2.2 Create dummies for categorical variables (drop one per category)
# ------------------------------------------------
# Identify categorical columns for dummy encoding
# Here: object or category dtype, plus any explicit categorical engineered columns.
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

# If there are ID-like columns you do NOT want to dummy, exclude them here
id_like_cols = ["order_no", "service_prime_line_no", "initial_ship_node"]  # adjust as needed
cat_cols_for_dummies = [c for c in cat_cols if c not in id_like_cols]

print("\nCategorical columns to dummy (excluding ID-like):")
print(cat_cols_for_dummies)

# Create dummies with drop_first=True to avoid dummy variable trap
df = pd.get_dummies(
    df,
    columns=cat_cols_for_dummies,
    drop_first=True
)

print("\nShape after dummy creation:", df.shape)


# ------------------------------------------------
# 2.3 Verify no perfect collinearity
# ------------------------------------------------

# 2.3.1 Drop any constant columns (no variance)
nunique = df.nunique()
constant_cols = nunique[nunique <= 1].index.tolist()

if constant_cols:
    print("\nDropping constant columns (no variance):")
    print(constant_cols)
    df = df.drop(columns=constant_cols)
else:
    print("\nNo constant columns found (good).")

# 2.3.2 Drop exactly duplicated columns (perfectly collinear)
# This is a strong check for perfect collinearity: two identical columns
# Note: this can be a bit memory-heavy if df has many columns, but for typical sizes it is fine.
duplicated_mask = df.T.duplicated()
duplicate_cols = df.columns[duplicated_mask].tolist()

if duplicate_cols:
    print("\nDropping duplicated columns (perfectly collinear with another column):")
    print(duplicate_cols)
    df = df.drop(columns=duplicate_cols)
else:
    print("\nNo duplicated columns found (good).")

print("\nFinal shape after STEP 2:", df.shape)


Object columns before numeric conversion: ['country_code', 'order_no', 'service_prime_line_no', 'initial_ship_node', 'ship_node_tier']

Dtypes after numeric conversion attempts:
country_code                                              object
order_no                                                   int64
service_prime_line_no                                      int64
first_event_time                             datetime64[us, UTC]
successful_AR                                              Int64
stock_z_score_imputed                                    float64
initial_ship_node                                         object
initial_no_order_lines                                     Int64
initial_ordered_qty                                      float64
initial_total_euros                                      float64
initial_click_collect_service_type                         Int64
initial_puop                                               Int64
initial_pup                               

In [23]:
import pandas as pd

# We assume `df` is your current cleaned dataframe with all columns listed in your index printout.

# ================================
# Define columns to keep for modeling
# ================================

# Meta / ID columns (kept for reference, NOT as model features later)
id_cols = [
    "order_no",
    "service_prime_line_no",
    "first_event_time",
]

# Target
target_col = ["successful_AR"]

# Timing & stock
timing_stock_cols = [
    "time_to_auto_recovery_hours",
    "hours_until_promised_delivery",
    # "hours_total_promised_lead_time"  # EXCLUDED from modeling to avoid perfect collinearity
    "stock_z_score_imputed",
]

# Order size / complexity
order_complexity_cols = [
    "initial_no_order_lines",
    "initial_ordered_qty",
    # "initial_total_euros"  # EXCLUDED from modeling as redundant with qty & lines
]

# HFB engineered features
hfb_engineered_cols = [
    "hfb_total_qty",
    "hfb_n_categories",
    "hfb_large_furniture_qty",
    "hfb_complex_items_qty",
    "hfb_textiles_decor_qty",
    "has_large_furniture",
    # Raw initial_*_qty columns are NOT included as model features
]

# Service level engineered features
service_level_cols = [
    "initial_sl_total_items",
    "pct_sl1_items",
    "pct_high_priority_items",
    "has_sl1_items",
    # Raw initial_sl1_items...sl4_items and initial_sl_item_is_null are excluded
]

# Service type dummies
service_type_cols = [
    "initial_click_collect_service_type",
    "initial_puop",
    "initial_pup",
    "initial_home_delivery_service_type",
    # One of these will be baseline implicitly in the model (do not worry here)
]

# Sales channel & rare indicator
sales_channel_cols = [
    "initial_store",
    "initial_phone",
    "initial_internet",
    "initial_remote_email_sales",
    "initial_remote_kitchen",
    "initial_sales_channel_other",
    "any_rare_binary_flag",
]

# Transport method
transport_cols = [
    "initial_parcel",
    "initial_truck",
    # depending on encoding, one can be baseline at modeling stage
]

# Geography and node tiers
country_cols = [
    "country_code_AU",
    "country_code_BE",
    "country_code_CA",
    "country_code_CH",
    "country_code_DE",
    "country_code_DK",
    "country_code_ES",
    "country_code_FI",
    "country_code_FR",
    "country_code_GB",
    "country_code_IE",
    "country_code_IN",
    "country_code_IT",
    "country_code_NL",
    "country_code_NO",
    "country_code_PL",
    "country_code_SE",
    "country_code_US",
]

ship_node_tier_cols = [
    "ship_node_tier_MEDIUM",
    "ship_node_tier_SMALL",
    # LARGE is implicit baseline when both are 0
]

# ================================
# Build final list and intersect with existing columns
# ================================

cols_proposed = (
    id_cols
    + target_col
    + timing_stock_cols
    + order_complexity_cols
    + hfb_engineered_cols
    + service_level_cols
    + service_type_cols
    + sales_channel_cols
    + transport_cols
    + country_cols
    + ship_node_tier_cols
)

# Keep only columns that actually exist in df
cols_existing = [c for c in cols_proposed if c in df.columns]

missing_cols = [c for c in cols_proposed if c not in df.columns]
if missing_cols:
    print("Warning: the following proposed columns were not found in df and are skipped:")
    print(missing_cols)

# Create df_modeling with the selected columns
df_modeling = df[cols_existing].copy()

print("df_modeling shape:", df_modeling.shape)
print("Columns in df_modeling:")
print(df_modeling.columns.tolist())


df_modeling shape: (32322, 52)
Columns in df_modeling:
['order_no', 'service_prime_line_no', 'first_event_time', 'successful_AR', 'time_to_auto_recovery_hours', 'hours_until_promised_delivery', 'stock_z_score_imputed', 'initial_no_order_lines', 'initial_ordered_qty', 'hfb_total_qty', 'hfb_n_categories', 'hfb_large_furniture_qty', 'hfb_complex_items_qty', 'hfb_textiles_decor_qty', 'has_large_furniture', 'initial_sl_total_items', 'pct_sl1_items', 'pct_high_priority_items', 'has_sl1_items', 'initial_click_collect_service_type', 'initial_puop', 'initial_pup', 'initial_home_delivery_service_type', 'initial_store', 'initial_phone', 'initial_internet', 'initial_remote_email_sales', 'initial_remote_kitchen', 'initial_sales_channel_other', 'any_rare_binary_flag', 'initial_parcel', 'initial_truck', 'country_code_AU', 'country_code_BE', 'country_code_CA', 'country_code_CH', 'country_code_DE', 'country_code_DK', 'country_code_ES', 'country_code_FI', 'country_code_FR', 'country_code_GB', 'country_c

In [26]:
import pandas as pd

# Standardise dtypes in df_modeling:
# - Convert pandas nullable Int64 -> int64 (if no NaNs) or float64 (if NaNs)
# - Convert pandas nullable Float64 -> float64
# - Leave existing int64/float64 as is

print("Dtypes before standardisation:")
print(df_modeling.dtypes)

for col in df_modeling.columns:
    dt = str(df_modeling[col].dtype)

    # Handle pandas nullable integer
    if dt == "Int64":
        if df_modeling[col].isna().any():
            # If there are NaNs, convert to float64 (can represent NaNs)
            df_modeling[col] = df_modeling[col].astype("float64")
        else:
            # No NaNs -> safe to convert to regular int64
            df_modeling[col] = df_modeling[col].astype("int64")

    # Handle pandas nullable float
    elif dt == "Float64":
        df_modeling[col] = df_modeling[col].astype("float64")

print("\nDtypes after standardisation:")
print(df_modeling.dtypes)


Dtypes before standardisation:
order_no                                            int64
service_prime_line_no                               int64
first_event_time                      datetime64[us, UTC]
successful_AR                                       Int64
time_to_auto_recovery_hours                       float64
hours_until_promised_delivery                     float64
stock_z_score_imputed                             float64
initial_no_order_lines                              Int64
initial_ordered_qty                               float64
hfb_total_qty                                     float64
hfb_n_categories                                    int64
hfb_large_furniture_qty                           float64
hfb_complex_items_qty                             float64
hfb_textiles_decor_qty                            float64
has_large_furniture                                 int64
initial_sl_total_items                              Int64
pct_sl1_items                            

# STEP 3: VIF Diagnostics 
* VIF on numerics only 
* Identify high VIF (> 5) 

In [31]:
import os
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

# =======================================
# Fix dtypes for dummy-like columns, then run VIF
# =======================================

# 1) Identify dummy-like columns: only 0/1 (ignoring NaN)
dummy_like_cols = []
for col in df_modeling.columns:
    uniques = df_modeling[col].dropna().unique()
    # Treat as dummy if all values are in {0, 1}
    if len(uniques) > 0 and set(uniques).issubset({0, 1}):
        dummy_like_cols.append(col)

print("Dummy-like columns (0/1 only):")
print(dummy_like_cols)

# 2) Cast all dummy-like columns to plain int64
for col in dummy_like_cols:
    df_modeling[col] = df_modeling[col].astype("int64")

# Optional: check dtypes for country_code_* and ship_node_tier_* now
print("\nDtypes for country_code_* and ship_node_tier_* after casting:")
mask_cc = df_modeling.columns.str.startswith("country_code_")
mask_sn = df_modeling.columns.str.startswith("ship_node_tier_")
print(df_modeling.loc[:, mask_cc | mask_sn].dtypes)

# =======================================
# STEP 3 (re-run): VIF Diagnostics (numerics only)
# =======================================

# Ensure OUTPUT_PATH exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ID and target to exclude from VIF
id_cols = [
    "order_no",
    "service_prime_line_no",
    "first_event_time",
]
target_col = ["successful_AR"]
exclude_from_vif = set(id_cols + target_col)

# 3) Select numeric columns (now including dummies cast to int64)
numeric_cols = df_modeling.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

numeric_cols_for_vif = [c for c in numeric_cols if c not in exclude_from_vif]

print("\nNumeric columns considered for VIF (after dtype fix):")
print(numeric_cols_for_vif)

if len(numeric_cols_for_vif) < 2:
    print("\nToo few numeric columns for VIF analysis. No Excel file will be created.")
else:
    # 4) Prepare matrix for VIF
    X_vif = df_modeling[numeric_cols_for_vif].dropna().copy()

    # 5) Compute VIF
    vif_data = pd.DataFrame()
    vif_data["variable"] = numeric_cols_for_vif
    vif_data["VIF"] = [
        variance_inflation_factor(X_vif.values, i)
        for i in range(len(numeric_cols_for_vif))
    ]
    vif_data = vif_data.sort_values("VIF", ascending=False).reset_index(drop=True)

    print("\nVIF results (sorted, high to low):")
    display(vif_data)

    # 6) Identify high VIF variables (> 5)
    high_vif_threshold = 5.0
    high_vif_df = vif_data[vif_data["VIF"] > high_vif_threshold].copy()

    if not high_vif_df.empty:
        print(f"\nVariables with VIF > {high_vif_threshold}:")
        display(high_vif_df)
    else:
        print(f"\nNo variables with VIF > {high_vif_threshold}.")

    # 7) Write VIF results to Excel
    vif_output_path = os.path.join(OUTPUT_PATH, "step3_vif_results_with_dummies.xlsx")

    with pd.ExcelWriter(vif_output_path, engine="openpyxl") as writer:
        vif_data.to_excel(
            writer,
            sheet_name="VIF_all_numeric",
            index=False
        )
        if not high_vif_df.empty:
            high_vif_df.to_excel(
                writer,
                sheet_name="VIF_high_only",
                index=False
            )

    print(f"\nVIF results written to: {vif_output_path}")


Dummy-like columns (0/1 only):
['successful_AR', 'has_large_furniture', 'has_sl1_items', 'initial_click_collect_service_type', 'initial_puop', 'initial_pup', 'initial_home_delivery_service_type', 'initial_store', 'initial_phone', 'initial_internet', 'initial_remote_email_sales', 'initial_remote_kitchen', 'initial_sales_channel_other', 'any_rare_binary_flag', 'initial_parcel', 'initial_truck', 'country_code_AU', 'country_code_BE', 'country_code_CA', 'country_code_CH', 'country_code_DE', 'country_code_DK', 'country_code_ES', 'country_code_FI', 'country_code_FR', 'country_code_GB', 'country_code_IE', 'country_code_IN', 'country_code_IT', 'country_code_NL', 'country_code_NO', 'country_code_PL', 'country_code_SE', 'country_code_US', 'ship_node_tier_MEDIUM', 'ship_node_tier_SMALL']

Dtypes for country_code_* and ship_node_tier_* after casting:
country_code_AU          int64
country_code_BE          int64
country_code_CA          int64
country_code_CH          int64
country_code_DE          i

,variable,VIF
0,hfb_total_qty,455.030931
1,initial_ordered_qty,435.220582
2,initial_internet,171.839197
3,initial_home_delivery_service_type,116.763674
4,initial_no_order_lines,38.150123
5,initial_store,29.304258
6,initial_parcel,26.982675
7,country_code_DE,25.466356
8,initial_sl_total_items,23.309029
9,country_code_US,22.137643



Variables with VIF > 5.0:


,variable,VIF
0,hfb_total_qty,455.030931
1,initial_ordered_qty,435.220582
2,initial_internet,171.839197
3,initial_home_delivery_service_type,116.763674
4,initial_no_order_lines,38.150123
5,initial_store,29.304258
6,initial_parcel,26.982675
7,country_code_DE,25.466356
8,initial_sl_total_items,23.309029
9,country_code_US,22.137643



VIF results written to: C:/Users/NILAV/OneDrive - IKEA/Documents/Project folder/NO STOCK business case/Casual Inference/step3_vif_results_with_dummies.xlsx


# STEP 4: Fix VIF
Remove or combine high VIF vars
Re-run VIF
Verify < 10 for all 

In [32]:
import os
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --------------------------------------
# 1) Define columns to drop based on VIF + domain logic
# --------------------------------------
cols_to_drop_for_vif = [
    # Size/complexity (highly redundant)
    "hfb_total_qty",
    "initial_sl_total_items",
    "hfb_complex_items_qty",
    # Channel/delivery (strongly collinear with service type / country)
    "initial_internet",
    "initial_store",
    "initial_phone",
]

# Keep only columns that actually exist
cols_to_drop_for_vif = [c for c in cols_to_drop_for_vif if c in df_modeling.columns]

print("Dropping these columns from VIF/modeling feature set (for now):")
print(cols_to_drop_for_vif)

# Create a reduced dataframe for VIF diagnostics
df_modeling_reduced = df_modeling.drop(columns=cols_to_drop_for_vif, errors="ignore").copy()

print("\nShape of df_modeling_reduced:", df_modeling_reduced.shape)

# --------------------------------------
# 2) Re-run VIF on numerics only
# --------------------------------------

# ID and target to exclude from VIF
id_cols = [
    "order_no",
    "service_prime_line_no",
    "first_event_time",
]
target_col = ["successful_AR"]
exclude_from_vif = set(id_cols + target_col)

# Ensure dummy-like columns are proper ints
dummy_like_cols = []
for col in df_modeling_reduced.columns:
    uniques = df_modeling_reduced[col].dropna().unique()
    if len(uniques) > 0 and set(uniques).issubset({0, 1}):
        dummy_like_cols.append(col)

for col in dummy_like_cols:
    df_modeling_reduced[col] = df_modeling_reduced[col].astype("int64")

# Select numeric columns
numeric_cols = df_modeling_reduced.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

numeric_cols_for_vif = [c for c in numeric_cols if c not in exclude_from_vif]

print("\nNumeric columns considered for VIF after dropping:")
print(numeric_cols_for_vif)

if len(numeric_cols_for_vif) < 2:
    print("\nToo few numeric columns for VIF analysis.")
else:
    X_vif = df_modeling_reduced[numeric_cols_for_vif].dropna().copy()

    vif_data = pd.DataFrame()
    vif_data["variable"] = numeric_cols_for_vif
    vif_data["VIF"] = [
        variance_inflation_factor(X_vif.values, i)
        for i in range(len(numeric_cols_for_vif))
    ]
    vif_data = vif_data.sort_values("VIF", ascending=False).reset_index(drop=True)

    print("\nVIF results after dropping high-VIF cluster variables:")
    display(vif_data)

    # Identify VIF > 10
    high_vif_threshold = 10.0
    high_vif_df = vif_data[vif_data["VIF"] > high_vif_threshold].copy()

    if not high_vif_df.empty:
        print(f"\nStill variables with VIF > {high_vif_threshold}:")
        display(high_vif_df)
    else:
        print(f"\nAll VIF <= {high_vif_threshold} now (good).")

    # Optionally write to Excel in same OUTPUT_PATH as before
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    output_path = os.path.join(OUTPUT_PATH, "step3_vif_results_after_reduction.xlsx")
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        vif_data.to_excel(writer, sheet_name="VIF_all_numeric", index=False)
        if not high_vif_df.empty:
            high_vif_df.to_excel(writer, sheet_name="VIF_high_only", index=False)

    print(f"\nVIF results written to: {output_path}")


Dropping these columns from VIF/modeling feature set (for now):
['hfb_total_qty', 'initial_sl_total_items', 'hfb_complex_items_qty', 'initial_internet', 'initial_store', 'initial_phone']

Shape of df_modeling_reduced: (32322, 46)

Numeric columns considered for VIF after dropping:
['time_to_auto_recovery_hours', 'hours_until_promised_delivery', 'stock_z_score_imputed', 'initial_no_order_lines', 'initial_ordered_qty', 'hfb_n_categories', 'hfb_large_furniture_qty', 'hfb_textiles_decor_qty', 'has_large_furniture', 'pct_sl1_items', 'pct_high_priority_items', 'has_sl1_items', 'initial_click_collect_service_type', 'initial_puop', 'initial_pup', 'initial_home_delivery_service_type', 'initial_remote_email_sales', 'initial_remote_kitchen', 'initial_sales_channel_other', 'any_rare_binary_flag', 'initial_parcel', 'initial_truck', 'country_code_AU', 'country_code_BE', 'country_code_CA', 'country_code_CH', 'country_code_DE', 'country_code_DK', 'country_code_ES', 'country_code_FI', 'country_code_FR'

,variable,VIF
0,initial_home_delivery_service_type,53.952942
1,initial_parcel,24.117860
2,country_code_DE,18.325132
3,country_code_US,15.731354
4,hfb_n_categories,7.812781
5,initial_ordered_qty,7.617576
6,initial_no_order_lines,7.378760
7,has_sl1_items,5.607226
8,initial_pup,4.300210
9,country_code_NL,3.986012



Still variables with VIF > 10.0:


,variable,VIF
0,initial_home_delivery_service_type,53.952942
1,initial_parcel,24.117860
2,country_code_DE,18.325132
3,country_code_US,15.731354



VIF results written to: C:/Users/NILAV/OneDrive - IKEA/Documents/Project folder/NO STOCK business case/Casual Inference/step3_vif_results_after_reduction.xlsx


## Reduced VIF-test

In [34]:
import os
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --------------------------------------
# 1) Drop initial_parcel from modeling feature set
# --------------------------------------
cols_to_drop_for_vif_2 = ["initial_parcel"]

cols_to_drop_for_vif_2 = [c for c in cols_to_drop_for_vif_2 if c in df_modeling_reduced.columns]

print("Dropping these additional columns from VIF/modeling feature set:")
print(cols_to_drop_for_vif_2)

df_modeling_reduced2 = df_modeling_reduced.drop(columns=cols_to_drop_for_vif_2, errors="ignore").copy()

print("\nShape of df_modeling_reduced2:", df_modeling_reduced2.shape)

# --------------------------------------
# 2) Re-run VIF on numerics only
# --------------------------------------

id_cols = ["order_no", "service_prime_line_no", "first_event_time"]
target_col = ["successful_AR"]
exclude_from_vif = set(id_cols + target_col)

# Ensure dummy-like columns are int64
dummy_like_cols = []
for col in df_modeling_reduced2.columns:
    uniques = df_modeling_reduced2[col].dropna().unique()
    if len(uniques) > 0 and set(uniques).issubset({0, 1}):
        dummy_like_cols.append(col)

for col in dummy_like_cols:
    df_modeling_reduced2[col] = df_modeling_reduced2[col].astype("int64")

numeric_cols = df_modeling_reduced2.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

numeric_cols_for_vif = [c for c in numeric_cols if c not in exclude_from_vif]

print("\nNumeric columns considered for VIF after second reduction:")
print(numeric_cols_for_vif)

if len(numeric_cols_for_vif) < 2:
    print("\nToo few numeric columns for VIF analysis.")
else:
    X_vif = df_modeling_reduced2[numeric_cols_for_vif].dropna().copy()

    vif_data2 = pd.DataFrame()
    vif_data2["variable"] = numeric_cols_for_vif
    vif_data2["VIF"] = [
        variance_inflation_factor(X_vif.values, i)
        for i in range(len(numeric_cols_for_vif))
    ]
    vif_data2 = vif_data2.sort_values("VIF", ascending=False).reset_index(drop=True)

    print("\nVIF results after dropping initial_parcel:")
    display(vif_data2)

    high_vif_threshold = 10.0
    high_vif_df2 = vif_data2[vif_data2["VIF"] > high_vif_threshold].copy()

    if not high_vif_df2.empty:
        print(f"\nStill variables with VIF > {high_vif_threshold}:")
        display(high_vif_df2)
    else:
        print(f"\nAll VIF <= {high_vif_threshold} now (good).")

    # Write to Excel if you want to track
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    output_path2 = os.path.join(OUTPUT_PATH, "step3_vif_results_after_reduction2.xlsx")
    with pd.ExcelWriter(output_path2, engine="openpyxl") as writer:
        vif_data2.to_excel(writer, sheet_name="VIF_all_numeric", index=False)
        if not high_vif_df2.empty:
            high_vif_df2.to_excel(writer, sheet_name="VIF_high_only", index=False)

    print(f"\nVIF results written to: {output_path2}")


Dropping these additional columns from VIF/modeling feature set:
['initial_parcel']

Shape of df_modeling_reduced2: (32322, 45)

Numeric columns considered for VIF after second reduction:
['time_to_auto_recovery_hours', 'hours_until_promised_delivery', 'stock_z_score_imputed', 'initial_no_order_lines', 'initial_ordered_qty', 'hfb_n_categories', 'hfb_large_furniture_qty', 'hfb_textiles_decor_qty', 'has_large_furniture', 'pct_sl1_items', 'pct_high_priority_items', 'has_sl1_items', 'initial_click_collect_service_type', 'initial_puop', 'initial_pup', 'initial_home_delivery_service_type', 'initial_remote_email_sales', 'initial_remote_kitchen', 'initial_sales_channel_other', 'any_rare_binary_flag', 'initial_truck', 'country_code_AU', 'country_code_BE', 'country_code_CA', 'country_code_CH', 'country_code_DE', 'country_code_DK', 'country_code_ES', 'country_code_FI', 'country_code_FR', 'country_code_GB', 'country_code_IE', 'country_code_IN', 'country_code_IT', 'country_code_NL', 'country_code_N

,variable,VIF
0,initial_home_delivery_service_type,46.242236
1,country_code_DE,17.408029
2,country_code_US,14.880265
3,initial_ordered_qty,7.617527
4,hfb_n_categories,7.583655
5,initial_no_order_lines,7.370319
6,has_sl1_items,5.432020
7,initial_pup,3.854077
8,country_code_NL,3.792633
9,pct_sl1_items,3.462382



Still variables with VIF > 10.0:


,variable,VIF
0,initial_home_delivery_service_type,46.242236
1,country_code_DE,17.408029
2,country_code_US,14.880265



VIF results written to: C:/Users/NILAV/OneDrive - IKEA/Documents/Project folder/NO STOCK business case/Casual Inference/step3_vif_results_after_reduction2.xlsx


## Create df_modeling_reduced (cleaned after VIF)

In [52]:
import pandas as pd

# We assume `df_modeling` exists and contains all relevant columns.

# ----------------------------------------
# 1) Define features to keep (from VIF table)
# ----------------------------------------
features_to_keep = [
    "initial_home_delivery_service_type",
    "country_code_DE",
    "country_code_US",
    "initial_ordered_qty",
    "hfb_n_categories",
    "initial_no_order_lines",
    "has_sl1_items",
    "initial_pup",
    "country_code_NL",
    "pct_sl1_items",
    "country_code_IT",
    "ship_node_tier_MEDIUM",
    "initial_truck",
    "hours_until_promised_delivery",
    "has_large_furniture",
    "country_code_PL",
    "country_code_SE",
    "hfb_large_furniture_qty",
    "pct_high_priority_items",
    "country_code_NO",
    "country_code_ES",
    "country_code_FR",
    "country_code_GB",
    "ship_node_tier_SMALL",
    "country_code_CA",
    "hfb_textiles_decor_qty",
    "country_code_BE",
    "time_to_auto_recovery_hours",
    "country_code_CH",
    "initial_puop",
    "country_code_IE",
    "country_code_AU",
    "country_code_DK",
    "country_code_IN",
    "initial_remote_kitchen",
    "country_code_FI",
    "initial_sales_channel_other",
    "initial_click_collect_service_type",
    "initial_remote_email_sales",
    "stock_z_score_imputed",
    "any_rare_binary_flag",
]

target_col = "successful_AR"

# ----------------------------------------
# 2) Build final column list for df_modeling_reduced
#    (target first, then features)
# ----------------------------------------
cols_requested = [target_col] + features_to_keep

# Keep only columns that actually exist in df_modeling
cols_existing = [c for c in cols_requested if c in df_modeling.columns]

# Warn if something is missing
missing_cols = [c for c in cols_requested if c not in df_modeling.columns]
if missing_cols:
    print("Warning: the following requested columns were not found in df_modeling and will be skipped:")
    print(missing_cols)

# ----------------------------------------
# 3) Create df_modeling_reduced
# ----------------------------------------
df_modeling_reduced = df_modeling[cols_existing].copy()

print("df_modeling_reduced shape:", df_modeling_reduced.shape)
print("Columns in df_modeling_reduced:")
print(df_modeling_reduced.columns.tolist())


df_modeling_reduced shape: (32322, 42)
Columns in df_modeling_reduced:
['successful_AR', 'initial_home_delivery_service_type', 'country_code_DE', 'country_code_US', 'initial_ordered_qty', 'hfb_n_categories', 'initial_no_order_lines', 'has_sl1_items', 'initial_pup', 'country_code_NL', 'pct_sl1_items', 'country_code_IT', 'ship_node_tier_MEDIUM', 'initial_truck', 'hours_until_promised_delivery', 'has_large_furniture', 'country_code_PL', 'country_code_SE', 'hfb_large_furniture_qty', 'pct_high_priority_items', 'country_code_NO', 'country_code_ES', 'country_code_FR', 'country_code_GB', 'ship_node_tier_SMALL', 'country_code_CA', 'hfb_textiles_decor_qty', 'country_code_BE', 'time_to_auto_recovery_hours', 'country_code_CH', 'initial_puop', 'country_code_IE', 'country_code_AU', 'country_code_DK', 'country_code_IN', 'initial_remote_kitchen', 'country_code_FI', 'initial_sales_channel_other', 'initial_click_collect_service_type', 'initial_remote_email_sales', 'stock_z_score_imputed', 'any_rare_bina

# STEP 5: Baseline model (stepwise)


In [45]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from collections import OrderedDict

# =====================================================================
# 0. CONFIG
# =====================================================================
OUTPUT_DIR = OUTPUT_PATH          # reuse your existing OUTPUT_PATH
TARGET_COL = "successful_AR"

# Timing variables (main variables of interest) – EXCLUDED in this step
TIMING_VARS = [
    "time_to_auto_recovery_hours",
    "hours_until_promised_delivery",
    "time_to_auto_recovery_hours_sq",
    "hours_until_promised_delivery_sq",
    "hours_total_promised_lead_time",
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =====================================================================
# 1. DEFINE VARIABLE GROUPS (from df_modeling_reduced)
#    – no timing vars here
# =====================================================================

# NOTE: adjust membership if your df_modeling_reduced differs slightly

groups = OrderedDict()

# 1) Order size & complexity & mix (non-timing)
groups["order_composition"] = [
    "initial_ordered_qty",
    "initial_no_order_lines",
    "hfb_n_categories",
    "hfb_large_furniture_qty",
    "hfb_textiles_decor_qty",
    "has_large_furniture",
]

# 2) Service level / priority
groups["service_level"] = [
    "has_sl1_items",
    "pct_sl1_items",
    "pct_high_priority_items",
]

# 3) Fulfilment / delivery type
groups["fulfilment_type"] = [
    "initial_home_delivery_service_type",
    "initial_pup",
    "initial_puop",
    "initial_truck",
]

# 4) Sales channel / contact channel
groups["sales_channel"] = [
    "initial_remote_kitchen",
    "initial_sales_channel_other",
    "initial_click_collect_service_type",
    "initial_remote_email_sales",
    "any_rare_binary_flag",
]

# 5) Geography (country dummies)
groups["country"] = [
    "country_code_DE",
    "country_code_US",
    "country_code_NL",
    "country_code_IT",
    "country_code_PL",
    "country_code_SE",
    "country_code_NO",
    "country_code_ES",
    "country_code_FR",
    "country_code_GB",
    "country_code_CA",
    "country_code_BE",
    "country_code_CH",
    "country_code_IE",
    "country_code_AU",
    "country_code_DK",
    "country_code_IN",
    "country_code_FI",
]

# 6) Node / network structure
groups["node_structure"] = [
    "ship_node_tier_MEDIUM",
    "ship_node_tier_SMALL",
]

# 7) Stock situation
groups["stock"] = [
    "stock_z_score_imputed",
]

# Filter out variables that do not actually exist in df_modeling_reduced
for gname, glist in groups.items():
    existing = [c for c in glist if c in df_modeling_reduced.columns and c not in TIMING_VARS]
    missing = [c for c in glist if c not in df_modeling_reduced.columns]
    groups[gname] = existing
    if missing:
        print(f"[{gname}] WARNING – not found in df_modeling_reduced and skipped:", missing)

print("\nFinal groups (after checking against df_modeling_reduced):")
for gname, glist in groups.items():
    print(f"  {gname} ({len(glist)} vars): {glist}")


# =====================================================================
# 2. GENERIC LOGIT RUNNER (reused for each step)
# =====================================================================

def run_logit_model(model_name, feature_list, df_source, target_col, output_path):
    os.makedirs(output_path, exist_ok=True)

    # 1. Build modeling dataset
    cols_needed = [target_col] + feature_list
    df_model = df_source[cols_needed].dropna().copy()
    print(f"\n[{model_name}] Dataset shape:", df_model.shape)

    y = df_model[target_col].astype(int)
    X = df_model[feature_list]
    X = sm.add_constant(X)

    # 2. Fit model
    logit_mod = sm.Logit(y, X)
    result = logit_mod.fit(disp=True)

    # 3. Coefficients, OR, marginal effects
    params = result.params
    bse = result.bse
    pvalues = result.pvalues
    conf_int = result.conf_int()
    conf_int.columns = ["ci_lower", "ci_upper"]

    coef_table = pd.concat([params, bse, pvalues, conf_int], axis=1)
    coef_table.columns = ["coef", "std_err", "p_value", "ci_lower", "ci_upper"]

    coef_table["odds_ratio"] = np.exp(coef_table["coef"])
    coef_table["or_ci_lower"] = np.exp(coef_table["ci_lower"])
    coef_table["or_ci_upper"] = np.exp(coef_table["ci_upper"])

    # Marginal effects (average)
    marg = result.get_margeff(at="overall")
    marg_df = marg.summary_frame()
    marg_cols = list(marg_df.columns)
    marg_df_renamed = marg_df.copy()

    if len(marg_cols) >= 6:
        marg_df_renamed = marg_df_renamed.rename(
            columns={
                marg_cols[0]: "marg_eff",
                marg_cols[1]: "marg_std_err",
                marg_cols[2]: "marg_z",
                marg_cols[3]: "marg_p_value",
                marg_cols[4]: "marg_ci_lower",
                marg_cols[5]: "marg_ci_upper",
            }
        )
        coef_table = coef_table.join(
            marg_df_renamed[
                ["marg_eff", "marg_std_err", "marg_z", "marg_p_value",
                 "marg_ci_lower", "marg_ci_upper"]
            ],
            how="left"
        )
    else:
        print(f"[{model_name}] Warning: Unexpected marginal effects columns:", marg_cols)

    # 4. Model-level statistics
    n_obs = int(result.nobs)
    llf = float(result.llf)
    llnull = float(result.llnull)
    aic = float(result.aic)
    bic = float(result.bic)
    pseudo_r2 = float(result.prsquared)
    llr_pvalue = float(result.llr_pvalue)
    df_model = int(result.df_model)
    df_resid = int(result.df_resid)
    converged = bool(result.mle_retvals.get("converged", True))
    n_iter = result.mle_retvals.get("iterations", np.nan)

    model_stats = pd.DataFrame(
        {
            "metric": [
                "model_name",
                "n_obs",
                "df_model",
                "df_resid",
                "log_likelihood",
                "log_likelihood_null",
                "pseudo_r2_mcfadden",
                "AIC",
                "BIC",
                "LLR_pvalue",
                "converged",
                "n_iterations",
            ],
            "value": [
                model_name,
                n_obs,
                df_model,
                df_resid,
                llf,
                llnull,
                pseudo_r2,
                aic,
                bic,
                llr_pvalue,
                converged,
                n_iter,
            ],
        }
    )

    # 5. Write per-model Excel
    model_output_path = os.path.join(output_path, f"model_{model_name}.xlsx")
    with pd.ExcelWriter(model_output_path, engine="openpyxl") as writer:
        coef_table.to_excel(writer, sheet_name="coefficients", index=True)
        model_stats.to_excel(writer, sheet_name="model_stats", index=False)
    print(f"[{model_name}] Excel written to: {model_output_path}")

    # 6. Append to model_registry.xlsx
    registry_path = os.path.join(output_path, "model_registry.xlsx")
    registry_row = pd.DataFrame(
        [
            {
                "model_name": model_name,
                "n_obs": n_obs,
                "df_model": df_model,
                "df_resid": df_resid,
                "log_likelihood": llf,
                "log_likelihood_null": llnull,
                "pseudo_r2_mcfadden": pseudo_r2,
                "AIC": aic,
                "BIC": bic,
                "LLR_pvalue": llr_pvalue,
                "converged": converged,
                "n_iterations": n_iter,
            }
        ]
    )

    if os.path.exists(registry_path):
        try:
            existing_registry = pd.read_excel(registry_path, sheet_name="models")
            registry_all = pd.concat([existing_registry, registry_row], ignore_index=True)
        except Exception:
            registry_all = registry_row.copy()
    else:
        registry_all = registry_row.copy()

    with pd.ExcelWriter(registry_path, engine="openpyxl") as writer:
        registry_all.to_excel(writer, sheet_name="models", index=False)
    print(f"[{model_name}] Registry updated at: {registry_path}")

    # 7. Text summary
    txt_summary_path = os.path.join(output_path, f"model_{model_name}.txt")
    summary_text = result.summary2().as_text()
    with open(txt_summary_path, "w", encoding="utf-8") as f:
        f.write(summary_text)
    print(f"[{model_name}] Text summary written to: {txt_summary_path}")

    # 8. Marginal effects summary
    txt_marg_path = os.path.join(output_path, f"model_{model_name}_marginal_effects.txt")
    marg_text = marg.summary().as_text()
    with open(txt_marg_path, "w", encoding="utf-8") as f:
        f.write(marg_text)
    print(f"[{model_name}] Marginal effects written to: {txt_marg_path}")

    # 9. Odds ratios table (text)
    txt_or_path = os.path.join(output_path, f"model_{model_name}_odds_ratios.txt")
    or_table = coef_table[["odds_ratio", "or_ci_lower", "or_ci_upper", "p_value"]].copy()
    or_table = or_table.rename(
        columns={
            "odds_ratio": "OR",
            "or_ci_lower": "OR_2.5%",
            "or_ci_upper": "OR_97.5%",
            "p_value": "p_value",
        }
    )
    with open(txt_or_path, "w", encoding="utf-8") as f:
        f.write("Odds Ratios (Logit model: " + model_name + ")\n")
        f.write("=" * 80 + "\n\n")
        f.write(or_table.to_string(float_format=lambda x: f"{x:0.6f}"))
        f.write("\n")
    print(f"[{model_name}] OR table written to: {txt_or_path}")

    return result, coef_table, model_stats


# =====================================================================
# 3. RUN SEQUENTIAL MODELS BY ADDING GROUPS ONE-BY-ONE
#    (NO TIMING VARIABLES YET)
# =====================================================================

current_features = []  # start with intercept-only model conceptually

for step_idx, (gname, glist) in enumerate(groups.items(), start=1):
    # Add this group's variables to the cumulative feature set
    current_features = current_features + glist

    model_name = f"step{step_idx}_{gname}"
    print(f"\n=== Running {model_name} ===")
    print("Features in model:", current_features)

    # Run model with all features up to and including this group
    result_step, coef_step, stats_step = run_logit_model(
        model_name=model_name,
        feature_list=current_features,
        df_source=df_modeling_reduced,
        target_col=TARGET_COL,
        output_path=OUTPUT_DIR,
    )

# Efter detta har du:
# - en modell per steg (step1_order_composition, step2_service_level, ...)
# - en rad per modell i model_registry.xlsx
# så du kan se hur AIC, BIC, pseudo R² etc ändras när varje grupp läggs till.
# Timing-variablerna lägger vi på senare, i en ny uppsättning modeller.



Final groups (after checking against df_modeling_reduced):
  order_composition (6 vars): ['initial_ordered_qty', 'initial_no_order_lines', 'hfb_n_categories', 'hfb_large_furniture_qty', 'hfb_textiles_decor_qty', 'has_large_furniture']
  service_level (3 vars): ['has_sl1_items', 'pct_sl1_items', 'pct_high_priority_items']
  fulfilment_type (4 vars): ['initial_home_delivery_service_type', 'initial_pup', 'initial_puop', 'initial_truck']
  sales_channel (5 vars): ['initial_remote_kitchen', 'initial_sales_channel_other', 'initial_click_collect_service_type', 'initial_remote_email_sales', 'any_rare_binary_flag']
  country (18 vars): ['country_code_DE', 'country_code_US', 'country_code_NL', 'country_code_IT', 'country_code_PL', 'country_code_SE', 'country_code_NO', 'country_code_ES', 'country_code_FR', 'country_code_GB', 'country_code_CA', 'country_code_BE', 'country_code_CH', 'country_code_IE', 'country_code_AU', 'country_code_DK', 'country_code_IN', 'country_code_FI']
  node_structure (2 v

# Full model (add key variables to baseline)

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm

def run_logit_model(model_name, feature_list, df_source, target_col, output_path):
    os.makedirs(output_path, exist_ok=True)

    # 1. Bygg modell-dataset
    cols_needed = [target_col] + feature_list
    df_model = df_source[cols_needed].copy()

    # Tvinga alla kolumner till numeriskt (fixar dtype=object-problem)
    for col in cols_needed:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

    # Debug: kolla om något fortfarande är object
    non_numeric = df_model.dtypes[df_model.dtypes == "object"].index.tolist()
    if non_numeric:
        print(f"[{model_name}] WARNING: fortfarande icke-numeriska kolumner:", non_numeric)

    # Droppa rader med NaN i target/features
    df_model = df_model.dropna(subset=cols_needed)
    print(f"\n[{model_name}] Dataset shape after numeric cast & dropna:", df_model.shape)

    y = df_model[target_col].astype(int)
    X = df_model[feature_list].astype(float)
    X = sm.add_constant(X)

    # 2. Fit model
    logit_mod = sm.Logit(y, X)
    result = logit_mod.fit(disp=True)

    # 3. Koef + OR + marginaleffekter
    params = result.params
    bse = result.bse
    pvalues = result.pvalues
    conf_int = result.conf_int()
    conf_int.columns = ["ci_lower", "ci_upper"]

    coef_table = pd.concat([params, bse, pvalues, conf_int], axis=1)
    coef_table.columns = ["coef", "std_err", "p_value", "ci_lower", "ci_upper"]

    coef_table["odds_ratio"] = np.exp(coef_table["coef"])
    coef_table["or_ci_lower"] = np.exp(coef_table["ci_lower"])
    coef_table["or_ci_upper"] = np.exp(coef_table["ci_upper"])

    # Marginaleffekter (average)
    marg = result.get_margeff(at="overall")
    marg_df = marg.summary_frame()
    marg_cols = list(marg_df.columns)
    marg_df_renamed = marg_df.copy()

    if len(marg_cols) >= 6:
        marg_df_renamed = marg_df_renamed.rename(
            columns={
                marg_cols[0]: "marg_eff",
                marg_cols[1]: "marg_std_err",
                marg_cols[2]: "marg_z",
                marg_cols[3]: "marg_p_value",
                marg_cols[4]: "marg_ci_lower",
                marg_cols[5]: "marg_ci_upper",
            }
        )
        coef_table = coef_table.join(
            marg_df_renamed[
                ["marg_eff", "marg_std_err", "marg_z", "marg_p_value",
                 "marg_ci_lower", "marg_ci_upper"]
            ],
            how="left"
        )
    else:
        print(f"[{model_name}] Warning: Unexpected marginal effects columns:", marg_cols)

    # 4. Modellstatistik
    n_obs = int(result.nobs)
    llf = float(result.llf)
    llnull = float(result.llnull)
    aic = float(result.aic)
    bic = float(result.bic)
    pseudo_r2 = float(result.prsquared)
    llr_pvalue = float(result.llr_pvalue)
    df_model_val = int(result.df_model)
    df_resid = int(result.df_resid)
    converged = bool(result.mle_retvals.get("converged", True))
    n_iter = result.mle_retvals.get("iterations", np.nan)

    model_stats = pd.DataFrame(
        {
            "metric": [
                "model_name",
                "n_obs",
                "df_model",
                "df_resid",
                "log_likelihood",
                "log_likelihood_null",
                "pseudo_r2_mcfadden",
                "AIC",
                "BIC",
                "LLR_pvalue",
                "converged",
                "n_iterations",
            ],
            "value": [
                model_name,
                n_obs,
                df_model_val,
                df_resid,
                llf,
                llnull,
                pseudo_r2,
                aic,
                bic,
                llr_pvalue,
                converged,
                n_iter,
            ],
        }
    )

    # 5. Excel per modell
    model_output_path = os.path.join(output_path, f"model_{model_name}.xlsx")
    with pd.ExcelWriter(model_output_path, engine="openpyxl") as writer:
        coef_table.to_excel(writer, sheet_name="coefficients", index=True)
        model_stats.to_excel(writer, sheet_name="model_stats", index=False)
    print(f"[{model_name}] Excel written to: {model_output_path}")

    # 6. Append till model_registry.xlsx
    registry_path = os.path.join(output_path, "model_registry.xlsx")
    registry_row = pd.DataFrame(
        [
            {
                "model_name": model_name,
                "n_obs": n_obs,
                "df_model": df_model_val,
                "df_resid": df_resid,
                "log_likelihood": llf,
                "log_likelihood_null": llnull,
                "pseudo_r2_mcfadden": pseudo_r2,
                "AIC": aic,
                "BIC": bic,
                "LLR_pvalue": llr_pvalue,
                "converged": converged,
                "n_iterations": n_iter,
            }
        ]
    )

    if os.path.exists(registry_path):
        try:
            existing_registry = pd.read_excel(registry_path, sheet_name="models")
            registry_all = pd.concat([existing_registry, registry_row], ignore_index=True)
        except Exception:
            registry_all = registry_row.copy()
    else:
        registry_all = registry_row.copy()

    with pd.ExcelWriter(registry_path, engine="openpyxl") as writer:
        registry_all.to_excel(writer, sheet_name="models", index=False)
    print(f"[{model_name}] Registry updated at: {registry_path}")

    # 7. TXT: huvudsummary (Logit)
    txt_summary_path = os.path.join(output_path, f"model_{model_name}.txt")
    summary_text = result.summary2().as_text()
    with open(txt_summary_path, "w", encoding="utf-8") as f:
        f.write(summary_text)
    print(f"[{model_name}] Text summary written to: {txt_summary_path}")

    # 8. TXT: marginaleffekter
    txt_marg_path = os.path.join(output_path, f"model_{model_name}_marginal_effects.txt")
    marg_text = marg.summary().as_text()
    with open(txt_marg_path, "w", encoding="utf-8") as f:
        f.write(marg_text)
    print(f"[{model_name}] Marginal effects summary written to: {txt_marg_path}")

    # 9. TXT: odds ratios
    txt_or_path = os.path.join(output_path, f"model_{model_name}_odds_ratios.txt")
    or_table = coef_table[["odds_ratio", "or_ci_lower", "or_ci_upper", "p_value"]].copy()
    or_table = or_table.rename(
        columns={
            "odds_ratio": "OR",
            "or_ci_lower": "OR_2.5%",
            "or_ci_upper": "OR_97.5%",
            "p_value": "p_value",
        }
    )
    with open(txt_or_path, "w", encoding="utf-8") as f:
        f.write("Odds Ratios (Logit model: " + model_name + ")\n")
        f.write("=" * 80 + "\n\n")
        f.write(or_table.to_string(float_format=lambda x: f"{x:0.6f}"))
        f.write("\n")
    print(f"[{model_name}] Odds ratios written to: {txt_or_path}")

    return result, coef_table, model_stats


In [54]:
# =============================
# FINAL MODEL: controls + timing
# =============================

TARGET_COL = "successful_AR"
OUTPUT_DIR = OUTPUT_PATH
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1) Skapa kvadrattermer för timing direkt i df_modeling_reduced
if "time_to_auto_recovery_hours" in df_modeling_reduced.columns:
    df_modeling_reduced["time_to_auto_recovery_hours_sq"] = (
        df_modeling_reduced["time_to_auto_recovery_hours"] ** 2
    )

if "hours_until_promised_delivery" in df_modeling_reduced.columns:
    df_modeling_reduced["hours_until_promised_delivery_sq"] = (
        df_modeling_reduced["hours_until_promised_delivery"] ** 2
    )

# 2) Definiera timingvariabler (huvudintresse)
timing_vars_linear = [
    "time_to_auto_recovery_hours",
    "hours_until_promised_delivery",
]

timing_vars_quad = [
    "time_to_auto_recovery_hours_sq",
    "hours_until_promised_delivery_sq",
]

timing_all = timing_vars_linear + timing_vars_quad + [
    "hours_total_promised_lead_time"  # om den mot förmodan finns
]

# 3) Kontrollfeatures = alla features utom timing
all_feature_cols = [c for c in df_modeling_reduced.columns if c != TARGET_COL]
control_features = [c for c in all_feature_cols if c not in timing_all]

print("Antal kontrollvariabler i basmodellen:", len(control_features))

# 4) Final feature-list: kontroller + timing (linjär + kvadrat)
final_features = control_features + timing_vars_linear + timing_vars_quad

print("Totalt antal features i slutmodellen:", len(final_features))
print("Timingvariabler i slutmodellen:", timing_vars_linear + timing_vars_quad)

# 5) Kör slutmodellen
MODEL_NAME = "final_controls_plus_timing"

result_final, coef_final, stats_final = run_logit_model(
    model_name=MODEL_NAME,
    feature_list=final_features,
    df_source=df_modeling_reduced,
    target_col=TARGET_COL,
    output_path=OUTPUT_DIR,
)


Antal kontrollvariabler i basmodellen: 39
Totalt antal features i slutmodellen: 43
Timingvariabler i slutmodellen: ['time_to_auto_recovery_hours', 'hours_until_promised_delivery', 'time_to_auto_recovery_hours_sq', 'hours_until_promised_delivery_sq']

[final_controls_plus_timing] Dataset shape after numeric cast & dropna: (32322, 44)
Optimization terminated successfully.
         Current function value: 0.577487
         Iterations 9
[final_controls_plus_timing] Excel written to: C:/Users/NILAV/OneDrive - IKEA/Documents/Project folder/NO STOCK business case/Casual Inference/model_final_controls_plus_timing.xlsx
[final_controls_plus_timing] Registry updated at: C:/Users/NILAV/OneDrive - IKEA/Documents/Project folder/NO STOCK business case/Casual Inference/model_registry.xlsx
[final_controls_plus_timing] Text summary written to: C:/Users/NILAV/OneDrive - IKEA/Documents/Project folder/NO STOCK business case/Casual Inference/model_final_controls_plus_timing.txt
[final_controls_plus_timing] 